### Setup

In [1]:
!ls

05-deployment-homework.ipynb  model.py		   pyproject.toml      train.py
Dockerfile.txt		      model_metadata.json  python-version.txt  uv.lock
__pycache__		      pipeline.bin	   q6_test.py
feature_defaults.json	      predict.py	   smoke_test.py


In [2]:
!which python

/opt/conda/envs/ml-zoomcamp/bin/python


In [3]:
!uv --version

uv 0.13.5 (x86_64-unknown-linux-gnu)


In [4]:
!uv sync --locked

Resolved 19 packages in 21ms
Checked 18 packages in 21ms


In [5]:
!sha256sum pipeline.bin

1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1  pipeline.bin


In [6]:
!uv run python smoke_test.py

{'conversion_probability': 0.769799, 'conversion': True}


### Question 1 — environment check

In [7]:
!uv --version

uv 0.13.5 (x86_64-unknown-linux-gnu)


### Question 2 — locked dependency

In [8]:
!grep -A 1 'scikit-learn==' pyproject.toml

    "scikit-learn==1.7.2",
    "uvicorn==0.38.0",


In [9]:
!grep -A 1 'name = "scikit-learn"' uv.lock

    { name = "scikit-learn" },
    { name = "uvicorn" },
--
    { name = "scikit-learn", specifier = "==1.7.2" },
    { name = "uvicorn", specifier = "==0.38.0" },
--
name = "scikit-learn"
version = "1.7.2"


### Question 3 — load the model

In [10]:
lead = {
    "lead_source": "paid_ads",
    "industry": "technology",
    "employment_status": "employed",
    "location": "north_america",
    "number_of_courses_viewed": 2,
    "annual_income": 79276.0,
    "interaction_count": 4,
    "lead_score": 0.41
}

In [12]:
import pickle

with open("pipeline.bin", "rb") as f_in:
  pipeline = pickle.load(f_in)

proba = pipeline.predict_proba(lead)[0,1]
print(f"probability: {proba:.3f}")

probability: 0.533


### Question 4 — serve the model

In [22]:
!uv run uvicorn predict:app --host 0.0.0.0 --port 9696

INFO:     Started server process [9370]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:9696 (Press CTRL+C to quit)
^C
INFO:     Shutting down
INFO:     Finished server process [9370]


In [24]:
lead_2 = {
    "lead_source": "organic_search",
    "industry": "technology",
    "employment_status": "employed",
    "location": "europe",
    "number_of_courses_viewed": 4,
    "annual_income": 80304.0,
    "interaction_count": 7,
    "lead_score": 0.74
}

In [36]:
import requests

test_url = 'http://localhost:9696/health'
url = 'http://localhost:9696/predict'

test_response = requests.get(test_url).json()
if test_response['status']=='ok':
  response = requests.post(url, json=lead_2).json()
  assert response['conversion_probability'] > 0 and response['conversion_probability'] < 1
  assert isinstance(response['conversion'], bool)
  conversion_proba = response['conversion_probability']
  print(f"conversion probability: {conversion_proba:.3f}")

conversion probability: 0.770


### Question 5 — look at the container configuration

In [7]:
!pwd
!ls -la

/workspaces/machine-learning-zoomcamp-hw/05-deployment-hw
total 104
drwxrwxrwx+  4 vscode vscode  4096 Oct 26 10:41 .
drwxrwxrwx+ 12 vscode root    4096 Oct 26 05:59 ..
drwxrwxrwx+  4 vscode vscode  4096 Oct 26 04:58 .venv
-rw-rw-rw-   1 vscode vscode  9295 Oct 26 05:48 05-deployment-homework.ipynb
-rw-rw-rw-   1 vscode vscode   598 Oct 26 03:49 Dockerfile
drwxrwxrwx+  2 vscode vscode  4096 Oct 26 05:14 __pycache__
-rw-rw-rw-   1 vscode vscode   116 Oct 26 03:49 feature_defaults.json
-rw-rw-rw-   1 vscode vscode  1456 Oct 26 03:49 model.py
-rw-rw-rw-   1 vscode vscode   872 Oct 26 03:49 model_metadata.json
-rw-rw-rw-   1 vscode vscode  1941 Oct 26 03:49 pipeline.bin
-rw-rw-rw-   1 vscode vscode  1426 Oct 26 03:49 predict.py
-rw-rw-rw-   1 vscode vscode   326 Oct 26 03:49 pyproject.toml
-rw-rw-rw-   1 vscode vscode     8 Oct 26 03:49 python-version
-rw-rw-rw-   1 vscode vscode   828 Oct 26 03:49 q6_test.py
-rw-rw-rw-   1 vscode vscode   777 Oct 26 03:49 smoke_test.py
-rw-rw-rw-   1 vsco

In [8]:
!find /workspaces/machine-learning-zoomcamp-hw -name "Dockerfile"

/workspaces/machine-learning-zoomcamp-hw/05-deployment-hw/Dockerfile


In [21]:
!sed -i 's/\r$//' Dockerfile .python-version pyproject.toml uv.lock

!docker build --network=host --progress=plain -t zoomcamp-model:2026-hw5 .

#0 building with "default" instance using docker driver

#1 [internal] load .dockerignore
#1 transferring context:
#1 transferring context: 46B done
#1 DONE 0.1s

#2 [internal] load build definition from Dockerfile
#2 transferring dockerfile: 751B done
#2 DONE 0.1s

#3 [internal] load metadata for ghcr.io/astral-sh/uv:0.10.11@sha256:3472e43b4e738cf911c99d41bb34331280efad54c73b1def654a6227bb59b2b4
#3 DONE 0.0s

#4 [internal] load metadata for docker.io/library/python:3.11.15-slim-bookworm@sha256:d29f48a31a8b408ed19272ca1e7b10ebae13b240a27e862d3d4217c528e2e0c3
#4 DONE 0.0s

#5 FROM ghcr.io/astral-sh/uv:0.10.11@sha256:3472e43b4e738cf911c99d41bb34331280efad54c73b1def654a6227bb59b2b4
#5 DONE 0.1s

#6 [stage-0 1/6] FROM docker.io/library/python:3.11.15-slim-bookworm@sha256:d29f48a31a8b408ed19272ca1e7b10ebae13b240a27e862d3d4217c528e2e0c3
#6 DONE 0.1s

#7 [internal] load build context
#7 transferring context: 32.91kB 0.0s done
#7 DONE 0.1s

#8 [stage-0 2/6] COPY --from=ghcr.io/astral-sh/uv:0.1

### Question 6 — run the container

In [22]:
!docker run --rm -p 9696:9696 zoomcamp-model:2026-hw5

^C


In [23]:
!python q6_test.py

{'conversion_probability': 0.769799, 'conversion': True}


In [24]:
!curl -s http://localhost:9696/health

{"status":"ok","model_sha256":"1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1","feature_defaults_sha256":"1694f842b908fac9f53c0cc19be6d91f934befc63af23cc32fefafde69acd0cd"}